## Accuracy assessment - setting the best threshold

In [ ]:
# importing packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc


In [ ]:
# paths and files
version = 1
# path = './bestThresholdCalcFiles/'
path = './thresholdsCalc/'

print('path ready!')

path ready!


In [ ]:
# functions
def calcAccUsingThreshold (df, thresh):

    # df['value'] = df['value'].astype(int)
    df['predicted'] = (df['classification'] >= thresh).astype(int)

    # labels
    trueLabels = df['value']
    predictedLabels = df['predicted']

    # Generate the confusion matrix
    tn, fp, fn, tp = confusion_matrix(trueLabels, predictedLabels, labels=[0, 1]).ravel()
    # print(tn, fp, fn, tp)

    # Sensitivity (Recall or True Positive Rate)
    sensitivity = tp / (tp + fn)

    # Specificity (True Negative Rate)
    specificity = tn / (tn + fp)

    # overall acc
    oa = accuracy_score(trueLabels, predictedLabels)

    return sensitivity, specificity, oa

# def accByCarta (year, carta, sample_sizes, bestThresh = True):
# def accByCarta (years, carta, bestThresh = True):
def accByCarta (carta, bestThresh = True):
    auc_results = []

    fileName = path + 'classification_s' + str(220) + '_' + carta + '_v' + str(version)

    # reading the files
    dfToCalcAcc = pd.read_csv(f'{fileName}.csv')

    available_years = sorted(dfToCalcAcc["year"].dropna().unique())

    for year in available_years:

        # # file name
        # fileName = path + 'classification_'+str(year)+'_'+carta+'_sSize_'+str(size)
        # fileName = path + 'teste_'+str(year)+'_'+carta+'_sSize_'+str(size)

        # version
        # classification_1985_SF_v1
        # fileName = path + 'classification_'+str(year)+'_'+carta+'_v'+str(version)
        # fileName = path + 'classification_'+str(year)+'_s400_'+carta+'_v'+str(version)
        # fileName = path + 'classification_'+str(year)+'_s450_'+carta+'_v'+str(version)
        # fileName = path + 'classification_'+str(year)+'_s220_'+carta+'_v'+str(version)
        # fileName = path + 'classification_'+str(year)+'_s2000_'+carta+'_v'+str(version)
        fileName = path + 'classification_s' + str(220) + '_' + carta + '_v' + str(version)

        # reading the files
        # data = pd.read_csv(f'{fileName}.csv')
        data = dfToCalcAcc[dfToCalcAcc['year'] == year]
        y_true = data['value']
        y_scores = data['classification']

        # print('y_true\n', y_true)
        # print('y_scores\n', y_scores);

        fpr, tpr, _ = roc_curve(y_true, y_scores)
        auc_value = auc(fpr, tpr)

        cutPercentile = data[data['value']==1].classification.quantile(0.15)*0.95

        if bestThresh == False:

            # best threshold defined as 50%
            bestJthreshold = cutPercentile
            # bestJthreshold = 50

        else:
            # computing j statistic
            youden_j = tpr - fpr

            # getting max j
            bestJindex = np.argmax(youden_j)
            bestJthreshold = _[bestJindex]

        # preparing the results
        sensitivity, specificity, oa = calcAccUsingThreshold (data, bestJthreshold)
        # auc_results.append((year, carta, auc_value, bestJthreshold, sensitivity, specificity, oa, cutPercentile))

        # appending the results
        auc_results.append((year, carta,
                            round(auc_value, 2),
                            round(bestJthreshold, 2),
                            round(sensitivity, 2),
                            round(specificity, 2),
                            round(oa, 2),
                            round(cutPercentile, 2)))


    # Reading the auc df
    dfauc = pd.DataFrame(auc_results, columns=["year", "carta", "AUC", "threshold", "sensitivity", "specificity", "oa", "cutPercentile"])

    # print('acc calculated!')
    return dfauc

# def accByList (cartas, year, bestThresh = True):
def accByList (cartas, bestThresh = True):
    dfToMerge = []

    for carta in cartas:
        try:
            # df = accByCarta (year, carta, bestThresh)
            df = accByCarta (carta, bestThresh)
            dfToMerge.append(df)
            # print("ok with " + carta)
        except:
            # print( "\'"+ carta ",\'// Error with " + carta + ", " + str(year))
            # print(f"\'{carta}\',// Error with {carta}, {year}")
            print(f"\'{carta}\',// Error with {carta}")

    # print(f'//--- end {str(year)} ---')
    # complete dataframe
    result = pd.concat(dfToMerge, ignore_index=True)

    return result

def checking (gridnames):

    # expected years
    expectedYears = set(range(1985,2025))

    # list the non-existent files
    results = []

    version = 1
    sSize = 220

    for gridname in gridnames:
        # classification_s220_NA-19-Y-D_v1
        filename = path + 'classification_s' + str(sSize) + '_' + gridname + '_v' + str(version) + '.csv'

        if not os.path.exists (filename):
            print(f'file not found: {gridname}')
            continue

        # read the csv file
        df = pd.read_csv(filename)

        # get the years present in the csv file
        existentYears = set(df['year'])

        # finding missing years
        # missingYears = expectedYears - existentYears
        missingYears = {year:('' if year in existentYears else 'x') for year in expectedYears}

        results.append({'gridname': gridname, **missingYears})

    df_results = pd.DataFrame(results)

    return df_results


In [ ]:
# grid list
gridList = [
  # gridset 1
  "NA-19-Z-B",
  "NA-19-Z-D",
  "NA-19-Z-C",
  "NA-19-Z-A",
  "NA-19-Y-D",
  "NA-19-Y-B",
  "NA-20-V-A",
  "NA-20-V-D",
  "NA-20-V-B",
  "NA-20-Y-D",
  "NA-20-Y-C",
  "NA-20-Y-A",
  "NA-20-Z-D",
  "NA-20-Z-B",
  "NA-20-X-B",
  "NA-20-X-A",
  "NA-20-X-C",
  "NA-20-Z-A",
  "NA-20-X-D",
  "NA-21-Z-C",
  "NA-21-Z-A",
  "NA-21-X-C",
  "NA-21-Y-D",
  "NA-21-Y-C",
  "NA-21-Y-A",
  "NA-21-V-C",
  "NA-21-V-A",
  "NA-21-Z-B",
  "NA-21-X-D",
  "NA-21-Z-D",
  "NA-22-V-B",
  "NA-22-X-C",
  "NA-22-Z-A",
  "NA-22-Z-C",
  "NA-22-Y-D",
  "NA-22-Y-C",
  "NA-22-Y-B",
  "NA-22-Y-A",
  "NA-22-V-D",
  "NB-20-Y-D",
  "NB-20-Y-C",
  "NB-20-Z-D",
  "NB-20-Z-B",
  "NB-20-Z-C",
  "NB-21-Y-C",
  "NB-22-Y-D",
  "SA-19-X-B",
  "SA-19-X-A",
  "SA-19-V-D",
  "SA-19-Y-B",
  "SA-19-Y-D",
  "SA-19-Z-A",
  "SA-19-Z-C",
  "SA-19-Z-D",
  "SA-19-Z-B",
  "SA-19-X-D",
  "SA-20-V-B",
  "SA-20-V-A",
  "SA-20-V-C",
  "SA-20-Y-A",
  "SA-20-V-D",
  "SA-20-Y-D",
  "SA-20-Y-C",
  "SA-20-X-B",
  "SA-20-X-A",
  "SA-20-X-C",
  "SA-20-X-D",
  "SA-20-Z-A",
  "SA-20-Z-B",
  "SA-20-Z-C",
  "SA-20-Z-D",
  "SA-21-X-A",
  "SA-21-V-B",
  "SA-21-V-A",
  "SA-21-Z-C",
  "SA-21-Z-A",
  "SA-21-Y-D",
  "SA-21-Y-B",
  "SA-21-Y-A",
  "SA-21-V-D",
  "SA-21-V-C",
  "SA-21-X-C",
  "SA-21-Z-D",
  "SA-21-Z-B",
  "SA-21-X-D",
  "SA-21-Y-C",
  "SA-22-X-B",
  "SA-22-X-A",
  "SA-22-V-B",
  "SA-22-V-A",
  "SA-22-V-C",
  "SA-22-Y-C",
  "SA-22-Y-A",
  "SA-22-V-D",
  "SA-22-Z-A",
  "SA-22-Z-C",
  "SA-22-X-C",

  # gridset 2
  "SA-22-X-D",
  "SA-22-Z-D",
  "SA-22-Z-B",
  "SA-22-Y-B",
  "SA-22-Y-D",
  "SA-23-V-A",
  "SA-23-V-C",
  "SA-23-Y-A",
  "SA-23-Y-C",
  "SA-23-Y-D",
  "SA-23-Z-C",
  "SA-23-Z-D",
  "SA-23-Z-B",
  "SA-23-X-C",
  "SA-23-Z-A",
  "SA-23-Y-B",
  "SA-23-V-D",
  "SA-23-V-B",
  "SA-24-Y-C",
  "SA-24-Y-D",
  "SA-24-Z-C",
  "SA-24-Y-B",
  "SA-24-Y-A",
  "SB-18-Z-B",
  "SB-18-X-D",
  "SB-18-Z-D",
  "SB-19-V-B",
  "SB-19-V-C",
  "SB-19-V-A",
  "SB-19-Z-C",
  "SB-19-Y-D",
  "SB-19-Z-D",
  "SB-19-Z-A",
  "SB-19-X-D",
  "SB-19-X-B",
  "SB-19-Y-B",
  "SB-19-Y-C",
  "SB-19-Y-A",
  "SB-20-V-B",
  "SB-20-V-A",
  "SB-20-V-D",
  "SB-20-Y-B",
  "SB-20-Y-D",
  "SB-20-Y-C",
  "SB-20-X-A",
  "SB-20-X-B",
  "SB-20-Z-A",
  "SB-20-Z-C",
  "SB-20-Z-D",
  "SB-20-Z-B",
  "SB-20-X-C",
  "SB-20-X-D",
  "SB-21-V-A",
  "SB-21-V-D",
  "SB-21-Y-A",
  "SB-21-Y-B",
  "SB-21-Y-C",
  "SB-21-Y-D",
  "SB-21-Z-A",
  "SB-21-Z-C",
  "SB-21-X-C",
  "SB-21-X-A",
  "SB-21-Z-D",
  "SB-21-Z-B",
  "SB-21-X-D",
  "SB-21-X-B",
  "SB-22-V-A",
  "SB-22-V-B",
  "SB-22-X-A",
  "SB-22-X-B",
  "SB-22-X-C",
  "SB-22-X-D",
  "SB-22-Z-D",
  "SB-22-Z-B",
  "SB-22-Z-A",
  "SB-22-Z-C",
  "SB-22-Y-D",
  "SB-22-Y-C",
  "SB-22-Y-B",
  "SB-22-Y-A",
  "SB-22-V-D",
  "SB-22-V-C",
  "SB-23-V-C",
  "SB-23-V-A",
  "SB-23-Y-A",
  "SB-23-Y-C",
  "SB-23-Z-D",
  "SB-23-Z-C",
  "SB-23-Y-D",
  "SB-23-Y-B",
  "SB-23-Z-A",
  "SB-23-Z-B",
  "SB-23-X-D",
  "SB-23-X-C",
  "SB-23-V-D",
  "SB-23-X-A",
  "SB-23-V-B",
  "SB-23-X-B",
  "SB-24-Y-C",
  "SB-24-V-C",

  # gridset 3
  "SB-24-V-A",
  "SB-24-V-B",
  "SB-24-V-D",
  "SB-24-Y-A",
  "SB-24-Y-B",
  "SB-24-Y-D",
  "SB-24-Z-C",
  "SB-24-Z-B",
  "SB-24-Z-D",
  "SB-24-X-B",
  "SB-24-X-A",
  "SB-24-X-C",
  "SB-24-Z-A",
  "SB-24-X-D",
  "SB-25-V-C",
  "SB-25-Y-A",
  "SB-25-Y-C",
  "SC-18-X-B",
  "SC-18-X-D",
  "SC-19-Y-D",
  "SC-19-Y-B",
  "SC-19-V-D",
  "SC-19-Z-A",
  "SC-19-X-C",
  "SC-19-Z-B",
  "SC-19-X-D",
  "SC-19-X-B",
  "SC-19-X-A",
  "SC-19-V-B",
  "SC-19-V-A",
  "SC-19-V-C",
  "SC-19-Z-C",
  "SC-20-V-C",
  "SC-20-V-D",
  "SC-20-V-B",
  "SC-20-Y-B",
  "SC-20-Y-D",
  "SC-20-Y-C",
  "SC-20-Y-A",
  "SC-20-X-A",
  "SC-20-X-C",
  "SC-20-X-D",
  "SC-20-Z-B",
  "SC-20-Z-D",
  "SC-20-Z-C",
  "SC-20-Z-A",
  "SC-21-V-C",
  "SC-21-V-B",
  "SC-21-V-D",
  "SC-21-Y-D",
  "SC-21-Z-C",
  "SC-21-Z-A",
  "SC-21-X-C",
  "SC-21-Y-B",
  "SC-21-Y-C",
  "SC-21-Y-A",
  "SC-21-X-B",
  "SC-21-X-D",
  "SC-21-Z-B",
  "SC-21-Z-D",
  "SC-22-X-B",
  "SC-22-X-A",
  "SC-22-V-B",
  "SC-22-V-A",
  "SC-22-V-C",
  "SC-22-Y-A",
  "SC-22-Y-C",
  "SC-22-Y-B",
  "SC-22-Y-D",
  "SC-22-Z-C",
  "SC-22-Z-A",
  "SC-22-X-C",
  "SC-22-V-D",
  "SC-22-X-D",
  "SC-22-Z-D",
  "SC-22-Z-B",
  "SC-23-V-C",
  "SC-23-V-A",
  "SC-23-Y-C",
  "SC-23-Z-C",
  "SC-23-Y-D",
  "SC-23-Z-A",
  "SC-23-Y-B",
  "SC-23-X-C",
  "SC-23-V-D",
  "SC-23-Z-B",
  "SC-23-Z-D",
  "SC-23-X-D",
  "SC-23-X-B",
  "SC-23-X-A",
  "SC-23-V-B",
  "SC-23-Y-A",
  "SC-24-X-D",
  "SC-24-Z-B",
  "SC-24-Z-D",
  "SC-24-X-B",

  # gridset 4
  "SC-24-X-A",
  "SC-24-X-C",
  "SC-24-V-B",
  "SC-24-V-D",
  "SC-24-Z-A",
  "SC-24-Y-B",
  "SC-24-Z-C",
  "SC-24-Y-D",
  "SC-24-Y-C",
  "SC-24-Y-A",
  "SC-24-V-C",
  "SC-24-V-A",
  "SC-25-V-A",
  "SC-25-V-C",
  "SD-20-V-B",
  "SD-20-Z-D",
  "SD-20-Z-B",
  "SD-20-X-B",
  "SD-20-X-D",
  "SD-20-X-C",
  "SD-20-X-A",
  "SD-21-X-B",
  "SD-21-X-D",
  "SD-21-Z-D",
  "SD-21-Z-B",
  "SD-21-Y-C",
  "SD-21-Y-D",
  "SD-21-Y-A",
  "SD-21-Y-B",
  "SD-21-V-D",
  "SD-21-X-C",
  "SD-21-Z-A",
  "SD-21-Z-C",
  "SD-21-X-A",
  "SD-21-V-B",
  "SD-21-V-C",
  "SD-22-V-A",
  "SD-22-V-C",
  "SD-22-V-B",
  "SD-22-V-D",
  "SD-22-X-A",
  "SD-22-X-B",
  "SD-22-X-C",
  "SD-22-X-D",
  "SD-22-Z-A",
  "SD-22-Z-B",
  "SD-22-Z-C",
  "SD-22-Z-D",
  "SD-22-Y-D",
  "SD-22-Y-C",
  "SD-22-Y-B",
  "SD-22-Y-A",
  "SD-23-X-B",
  "SD-23-X-D",
  "SD-23-Z-B",
  "SD-23-Z-D",
  "SD-23-Z-A",
  "SD-23-Z-C",
  "SD-23-Y-D",
  "SD-23-Y-B",
  "SD-23-X-C",
  "SD-23-V-D",
  "SD-23-V-B",
  "SD-23-X-A",
  "SD-23-V-A",
  "SD-23-Y-A",
  "SD-23-V-C",
  "SD-23-Y-C",
  "SD-24-X-A",
  "SD-24-V-B",
  "SD-24-X-C",
  "SD-24-Z-C",
  "SD-24-Y-B",
  "SD-24-Z-A",
  "SD-24-V-D",
  "SD-24-V-A",
  "SD-24-V-C",
  "SD-24-Y-A",
  "SD-24-Y-C",
  "SD-24-Y-D",
  "SE-20-X-B",
  "SE-21-X-B",
  "SE-21-X-A",
  "SE-21-V-B",
  "SE-21-V-A",
  "SE-21-X-D",
  "SE-21-Z-B",
  "SE-21-Y-B",
  "SE-21-V-D",
  "SE-21-Y-D",
  "SE-21-Z-D",
  "SE-22-X-B",
  "SE-22-X-A",
  "SE-22-V-B",
  "SE-22-V-A",
  "SE-22-X-D",
  "SE-22-X-C",
  "SE-22-V-D",
  "SE-22-Y-B",

  # gridset 5
  "SE-22-Y-A",
  "SE-22-V-C",
  "SE-22-Y-C",
  "SE-22-Z-C",
  "SE-22-Z-A",
  "SE-22-Y-D",
  "SE-22-Z-B",
  "SE-22-Z-D",
  "SE-23-V-B",
  "SE-23-V-A",
  "SE-23-X-A",
  "SE-23-X-B",
  "SE-23-Y-D",
  "SE-23-Z-C",
  "SE-23-Z-D",
  "SE-23-Z-A",
  "SE-23-Y-B",
  "SE-23-X-C",
  "SE-23-V-D",
  "SE-23-X-D",
  "SE-23-Z-B",
  "SE-23-V-C",
  "SE-23-Y-A",
  "SE-23-Y-C",
  "SE-24-X-A",
  "SE-24-V-B",
  "SE-24-V-A",
  "SE-24-Y-C",
  "SE-24-V-C",
  "SE-24-Y-B",
  "SE-24-V-D",
  "SE-24-Y-D",
  "SE-24-Y-A",
  "SF-21-Z-A",
  "SF-21-Z-B",
  "SF-21-X-C",
  "SF-21-X-D",
  "SF-21-Y-B",
  "SF-21-V-D",
  "SF-21-V-B",
  "SF-21-X-A",
  "SF-21-Z-C",
  "SF-21-Z-D",
  "SF-21-X-B",
  "SF-22-V-A",
  "SF-22-V-C",
  "SF-22-Y-A",
  "SF-22-V-D",
  "SF-22-V-B",
  "SF-22-Y-B",
  "SF-22-X-C",
  "SF-22-Z-A",
  "SF-22-X-A",
  "SF-22-X-B",
  "SF-22-X-D",
  "SF-22-Z-B",
  "SF-22-Y-C",
  "SF-22-Y-D",
  "SF-22-Z-C",
  "SF-22-Z-D",
  "SF-23-Z-B",
  "SF-23-X-D",
  "SF-23-X-C",
  "SF-23-V-D",
  "SF-23-X-A",
  "SF-23-V-B",
  "SF-23-V-A",
  "SF-23-V-C",
  "SF-23-Y-A",
  "SF-23-Z-C",
  "SF-23-Z-D",
  "SF-23-Y-C",
  "SF-23-Z-A",
  "SF-23-Y-B",
  "SF-23-X-B",
  "SF-23-Y-D",
  "SF-24-Y-A",
  "SF-24-V-A",
  "SF-24-V-B",
  "SF-24-V-C",
  "SF-24-Y-C",
  "SG-21-Z-D",
  "SG-21-X-D",
  "SG-21-X-B",
  "SG-22-X-A",
  "SG-22-X-B",
  "SG-22-X-C",
  "SG-22-X-D",
  "SG-22-Z-B",
  "SG-22-Z-D",
  "SG-22-Z-A",
  "SG-22-Y-D",
  "SG-22-Y-B",
  "SG-22-Y-C",
  "SG-22-Y-A",
  "SG-22-V-D",
  "SG-22-V-B",
  "SG-22-V-A",

  # gridset 5
  "SG-22-V-C",
  "SG-22-Z-C",
  "SG-23-V-B",
  "SG-23-V-A",
  "SG-23-V-C",
  "SH-21-Y-B",
  "SH-21-V-D",
  "SH-21-X-A",
  "SH-21-X-C",
  "SH-21-X-D",
  "SH-21-X-B",
  "SH-21-Z-B",
  "SH-21-Z-D",
  "SH-21-Z-C",
  "SH-21-Z-A",
  "SH-22-X-A",
  "SH-22-V-B",
  "SH-22-V-A",
  "SH-22-V-C",
  "SH-22-Y-A",
  "SH-22-Y-C",
  "SH-22-Z-C",
  "SH-22-Z-A",
  "SH-22-X-C",
  "SH-22-V-D",
  "SH-22-X-B",
  "SH-22-X-D",
  "SH-22-Y-D",
  "SH-22-Y-B",
  "SI-22-V-A",
  "SI-22-V-C",
  "SI-22-V-B"
]


In [ ]:
# --------
# checking
absentData = checking (gridList)
absentData.to_csv('./thresholdDataAbcence_col10.csv')
print('abcent data checked!')

abcent data checked!


In [ ]:
# list of years
yearsList = list(range(1985,2025))
bestThresh = True

# calculating the accuracy into a dataframe
# dfauc = accByList (gridList, yearsList, bestThresh)
dfauc = accByList (gridList, bestThresh)

dfauc.to_csv('./thresholdDataCalculatedByGrid_col10.csv')
print('thresholds were exported!')

In [ ]:
# reading the csv of thresholds
dfWithThresholds = pd.read_csv('./thresholdDataCalculatedByGrid_col10.csv').drop(columns=['Unnamed: 0'], index=1)
dfWithThresholds['year'] = dfWithThresholds['year'].astype(int)
dfWithThresholds

,year,carta,AUC,threshold,sensitivity,specificity,oa,cutPercentile
0,1985,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,57.95
2,1987,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,70.30
3,1988,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,72.06
4,1989,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,77.90
5,1990,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,74.62
...,...,...,...,...,...,...,...,...
20044,2020,SI-22-V-B,0.99,41.0,0.95,0.96,0.96,59.85
20045,2021,SI-22-V-B,0.99,35.0,0.97,0.94,0.95,58.90
20046,2022,SI-22-V-B,0.99,33.0,0.96,0.93,0.94,54.15
20047,2023,SI-22-V-B,0.98,31.0,0.96,0.92,0.94,54.15


In [ ]:
# list of complete years
yearsRange = pd.DataFrame({'year': range(1985, 2025)})

# # # grid set to test
# gridList = [
#     'NA-19-Z-D'
# ]

# full set of combinations
indexComplete = pd.MultiIndex.from_product([gridList, yearsRange['year']], names=['carta', 'year'])
indexCompleteDf = pd.DataFrame(index=indexComplete).reset_index()

# merging with original dataset
indexCompleteDfMerged = indexCompleteDf.merge(dfWithThresholds, on=['year', 'carta'], how='left')

# Replace 'inf' values with NaN first (to handle them properly)
indexCompleteDfMerged.replace([np.inf, -np.inf], np.nan, inplace=True)

# # filling values for threshold where they empty
indexCompleteDfMerged['threshold'] = indexCompleteDfMerged['threshold'].fillna(round(indexCompleteDfMerged['cutPercentile']/1.25, 0))

# filling the remaining missing values
indexCompleteDfMerged[['threshold', 'cutPercentile']] = indexCompleteDfMerged[['threshold', 'cutPercentile']].fillna(99)

indexCompleteDfMerged.to_csv('./thresholdDataCalculatedByGrid_col10_filled.csv', index=None)
print('thresholds filled were exported!')

indexCompleteDfMerged

thresholds filled were exported!


,carta,year,AUC,threshold,sensitivity,specificity,oa,cutPercentile
0,NA-19-Z-B,1985,NaN,99.0,NaN,NaN,NaN,99.00
1,NA-19-Z-B,1986,NaN,99.0,NaN,NaN,NaN,99.00
2,NA-19-Z-B,1987,NaN,99.0,NaN,NaN,NaN,99.00
3,NA-19-Z-B,1988,NaN,99.0,NaN,NaN,NaN,99.00
4,NA-19-Z-B,1989,NaN,99.0,NaN,NaN,NaN,99.00
...,...,...,...,...,...,...,...,...
20875,SI-22-V-B,2020,0.99,41.0,0.95,0.96,0.96,59.85
20876,SI-22-V-B,2021,0.99,35.0,0.97,0.94,0.95,58.90
20877,SI-22-V-B,2022,0.99,33.0,0.96,0.93,0.94,54.15
20878,SI-22-V-B,2023,0.98,31.0,0.96,0.92,0.94,54.15


In [ ]:
# reding the final file of thresholds by year
# df_filled =
# indexCompleteDfMerged.to_csv('./thresholdDataCalculatedByGrid_col10_filled.csv', index=None)
# reading the csv of thresholds
dfWithThresholds = pd.read_csv('./thresholdDataCalculatedByGrid_col10.csv').drop(columns=['Unnamed: 0'], index=1)
dfWithThresholds['year'] = dfWithThresholds['year'].astype(int)
dfWithThresholds

,year,carta,AUC,threshold,sensitivity,specificity,oa,cutPercentile
0,1985,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,57.95
2,1987,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,70.30
3,1988,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,72.06
4,1989,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,77.90
5,1990,NA-19-Z-D,NaN,inf,0.00,NaN,0.00,74.62
...,...,...,...,...,...,...,...,...
20044,2020,SI-22-V-B,0.99,41.0,0.95,0.96,0.96,59.85
20045,2021,SI-22-V-B,0.99,35.0,0.97,0.94,0.95,58.90
20046,2022,SI-22-V-B,0.99,33.0,0.96,0.93,0.94,54.15
20047,2023,SI-22-V-B,0.98,31.0,0.96,0.92,0.94,54.15


In [ ]:
# list of complete years
yearsRange = pd.DataFrame({'year': range(1985, 2025)})

# full set of combinations
indexComplete = pd.MultiIndex.from_product([gridList, yearsRange['year']], names=['carta', 'year'])
indexCompleteDf = pd.DataFrame(index=indexComplete).reset_index()

# merging with original dataset
indexCompleteDfMerged = indexCompleteDf.merge(dfWithThresholds, on=['year', 'carta'], how='left')

# Replace 'inf' values with NaN first (to handle them properly)
indexCompleteDfMerged.replace([np.inf, -np.inf], np.nan, inplace=True)

# # filling values for threshold where they empty
indexCompleteDfMerged['threshold'] = indexCompleteDfMerged['threshold'].fillna(round(indexCompleteDfMerged['cutPercentile']/1.25, 0))

# setting the values aggregated by mean
result = (
    indexCompleteDfMerged.replace([np.inf, -np.inf], np.nan)
    .groupby('carta')
    .agg({
        'threshold': 'mean',
        'cutPercentile': 'mean',
    })
    .round(1)
    .reset_index()
    .fillna(99)
)

result = pd.DataFrame(result)

result.to_csv('./thresholdDataCalculatedByGrid_col10_filled_mean.csv', index=None)
print('thresholds filled with mean were exported!')

# # filling the remaining missing values
# indexCompleteDfMerged[['threshold', 'cutPercentile']] = indexCompleteDfMerged[['threshold', 'cutPercentile']].fillna(99)

# indexCompleteDfMerged.to_csv('./thresholdDataCalculatedByGrid_col10_filled.csv', index=None)
# print('thresholds filled were exported!')

# indexCompleteDfMerged

# # setting the values aggregated by mean
# result = (
#     dfWithThresholds.replace([np.inf, -np.inf], np.nan)
#     .groupby('carta')
#     .agg({
#         'threshold': 'mean',
#         'cutPercentile': 'mean',
#     })
#     .round(1)
#     .reset_index()
#     .fillna(99)
# )

# result = pd.DataFrame(result)
# result.to_csv('./')

thresholds filled with mean were exported!


In [ ]:
result

,carta,threshold,cutPercentile
0,NA-19-Y-B,99.0,99.0
1,NA-19-Y-D,99.0,99.0
2,NA-19-Z-A,99.0,99.0
3,NA-19-Z-B,99.0,99.0
4,NA-19-Z-C,49.8,79.9
...,...,...,...
517,SH-22-Z-A,35.2,50.5
518,SH-22-Z-C,39.4,50.8
519,SI-22-V-A,34.2,50.2
520,SI-22-V-B,35.6,51.0
